# Week 6 · Capstone — End-to-End Hardware-Friendly LLM

**Repository:** `bitwise-llm-forge` · **Theory doc:** [`docs/theory/week6_capstone.md`](../docs/theory/week6_capstone.md)

---

### What we'll assemble

A miniature transformer that combines **every** technique from weeks 1–5:

| Week | Component | Used as |
|---|---|---|
| 1 | `BitLinear` (ternary weights) | All four linear projections per block + LM head |
| 4 | Tiled / SDPA attention | The attention operator |
| 5 | `RotaryPositionalEmbedding` | Positional encoding in attention |
| 6 | `SwiGLU` + `RMSNorm` | FFN and normalization |

### Goals

1. Show that the assembled `MiniLLM` trains end-to-end with stable loss.
2. Verify it can **generate** coherent (within its training distribution) sequences.
3. Profile its **parameter count, on-disk size, and inference latency**, contrasted with an FP32 baseline.
4. Run an ablation: BitLinear ↔ FP32 linear, with everything else fixed.


In [ ]:
# Make ``src/`` importable when this notebook is launched from anywhere.
from pathlib import Path
import sys

_here = Path.cwd()
for candidate in (_here, *_here.parents):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from src.utils.seeding import set_seed
set_seed(42)

print("environment ready · torch", __import__("torch").__version__)


## 1 · Architecture overview

```
Token IDs ──► Embedding ─┐
                         ▼
   ┌──────────── × L blocks ────────────┐
   │  RMSNorm                            │
   │    ├─ BitLinear(Q,K,V)              │
   │    ├─ RoPE on Q, K                  │
   │    └─ scaled_dot_product_attention  │
   │    BitLinear(O)  + residual         │
   │  RMSNorm                            │
   │    ├─ BitLinear(w1, w3) (SwiGLU)    │
   │    └─ BitLinear(w2)     + residual  │
   └─────────────────────────────────────┘
                         │
                         ▼
                     RMSNorm
                         │
                         ▼
                BitLinear(d → V)     (tied to embedding by default)
                         │
                         ▼
                       Logits
```

All sub-modules come from `src/`. The full assembled model lives in `src/model/mini_llm.py`.


## 2 · Build a tiny instance and inspect it

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time, math, json
from pathlib import Path
import matplotlib.pyplot as plt

from src.model import MiniLLM, MiniLLMConfig

torch.manual_seed(0)

# A small but realistically-shaped instance so the notebook trains in seconds.
cfg = MiniLLMConfig(
    vocab_size=256,
    d_model=128,
    n_layers=2,
    n_heads=4,
    max_seq_len=256,
    ffn_mult=4,
    tie_word_embeddings=True,
)
model = MiniLLM(cfg)

def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"vocab_size  = {cfg.vocab_size}")
print(f"d_model     = {cfg.d_model}")
print(f"n_layers    = {cfg.n_layers}")
print(f"n_heads     = {cfg.n_heads}")
print(f"max_seq_len = {cfg.max_seq_len}")
print(f"\ntotal trainable params: {count_params(model):,}")
print(f"approx ternary disk size: {count_params(model) * 1.585 / 8 / 1024:.1f} KB")
print(f"approx fp32   disk size: {count_params(model) * 4 / 1024:.1f} KB")


## 3 · Sanity check the forward / generate path

In [ ]:
ids = torch.randint(0, cfg.vocab_size, (2, 16))
with torch.no_grad():
    logits = model(ids)
print("logits shape:", tuple(logits.shape))
assert logits.shape == (2, 16, cfg.vocab_size)

# Generation
prompt = torch.randint(0, cfg.vocab_size, (1, 4))
with torch.no_grad():
    cont = model.generate(prompt, max_new_tokens=8, temperature=1.0)
print("prompt:    ", prompt.tolist())
print("generated:", cont.tolist())
assert cont.shape == (1, 4 + 8)


## 4 · Train on a deterministic synthetic language

To validate that the architecture actually *learns*, we use a controlled language: a small alphabet whose next-token distribution depends on the previous 3 tokens via a fixed transition rule. A model that learns the rule should bring the loss well below the uniform-random baseline of $\log V$.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

V = cfg.vocab_size

def make_corpus(n: int) -> torch.Tensor:
    """A 3-gram language defined by a small hash function instead of a
    full V³ lookup table (which would need ~128 MB for V=256).
    Deterministic, periodic, and learnable by a 2-layer transformer."""
    g = torch.Generator().manual_seed(123)
    out = torch.zeros(n, dtype=torch.long)
    out[:3] = torch.randint(0, V, (3,), generator=g)
    # Small set of mixing constants — keeps the dependency on previous tokens
    # rich enough that the model has to learn 3-gram statistics.
    A, B, C, D = 17, 31, 53, 97
    for t in range(3, n):
        out[t] = (A * out[t-1].item() + B * out[t-2].item() + C * out[t-3].item() + D) % V
    return out


corpus = make_corpus(20_000)
print("corpus length:", len(corpus))
print("baseline (uniform-random) NLL:", math.log(V))

uniform_ppl = V
print("baseline (uniform-random) perplexity:", uniform_ppl)


In [ ]:
from src.utils.seeding import set_seed

def fresh_model() -> MiniLLM:
    return MiniLLM(cfg).to(device)


def train(model: MiniLLM, steps: int, *, batch: int = 32, ctx: int = 128,
          lr: float = 3e-3) -> tuple[list[int], list[float]]:
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    stream = corpus.to(device)
    history_x, history_y = [], []
    for step in range(steps):
        starts = torch.randint(0, len(stream) - ctx - 1, (batch,))
        idx = starts[:, None] + torch.arange(ctx + 1)[None, :]
        chunk = stream[idx]
        ids, targets = chunk[:, :-1], chunk[:, 1:]
        logits = model(ids)
        loss = F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step(); sched.step()
        if step % 20 == 0:
            history_x.append(step); history_y.append(float(loss.item()))
    return history_x, history_y


steps = 400 if device.type == "cuda" else 200
print(f"training the full BitLinear MiniLLM for {steps} steps...")
set_seed(0)
model_bit = fresh_model()
xs_bit, ys_bit = train(model_bit, steps=steps)
print(f"final BitLinear loss: {ys_bit[-1]:.4f}")


## 5 · Ablation: BitLinear ↔ FP32 nn.Linear

In [ ]:
# Make a copy of MiniLLM with all BitLinear replaced by nn.Linear.
# We do it surgically with a monkeypatch on the BitLinear factory used inside
# mini_llm.py. Easiest path: monkey-patch src.model.mini_llm.BitLinear before
# constructing.

import src.model.mini_llm as _mm

class _FPLinear(nn.Module):
    """Drop-in stand-in matching BitLinear's signature & API but using fp32."""
    def __init__(self, in_features: int, out_features: int, *, bias: bool = False,
                 activation_bits: int = 8, eps: float = 1e-5) -> None:
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.norm = nn.LayerNorm(in_features)
    def forward(self, x):
        x = self.norm(x)
        y = F.linear(x, self.weight)
        return y if self.bias is None else y + self.bias


# Build the FP variant
_original = _mm.BitLinear
_mm.BitLinear = _FPLinear
set_seed(0)
model_fp = MiniLLM(cfg).to(device)
_mm.BitLinear = _original                 # restore for future cells

print(f"FP32 variant params : {count_params(model_fp):,}")
print(f"BitLinear params    : {count_params(model_bit):,}")

print("\ntraining the FP32 variant for the same number of steps...")
xs_fp, ys_fp = train(model_fp, steps=steps)
print(f"final FP32 loss: {ys_fp[-1]:.4f}")


In [ ]:
# Plot side-by-side
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(xs_bit, ys_bit, label="BitLinear MiniLLM (ternary)", linewidth=2)
ax.plot(xs_fp,  ys_fp,  label="FP32 MiniLLM (baseline)",     linewidth=2, linestyle="--")
ax.axhline(math.log(V), color="grey", linestyle=":", label=f"uniform random = {math.log(V):.3f}")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy loss")
ax.set_title("Capstone training — ternary vs fp32 weights")
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Persist
Path("../benchmarks/results").mkdir(parents=True, exist_ok=True)
Path("../benchmarks/results/capstone_training.json").write_text(json.dumps({
    "config": cfg.__dict__,
    "steps": steps,
    "bit": list(zip(xs_bit, ys_bit)),
    "fp":  list(zip(xs_fp,  ys_fp)),
}, indent=2))
print("\nresults saved to ../benchmarks/results/capstone_training.json")


## 6 · Inference latency and disk-size scoreboard

In [ ]:
def median_time(fn, warmup: int = 3, iters: int = 10) -> float:
    for _ in range(warmup): fn()
    if device.type == "cuda": torch.cuda.synchronize()
    samples = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        if device.type == "cuda": torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)
    samples.sort(); return samples[len(samples) // 2]


prompt = torch.randint(0, V, (1, 32), device=device)
def fwd_bit(): return model_bit(prompt)
def fwd_fp():  return model_fp(prompt)

with torch.no_grad():
    t_bit = median_time(fwd_bit)
    t_fp  = median_time(fwd_fp)

# Approximate disk sizes
n = count_params(model_bit)
disk_bit_kb = n * 1.585 / 8 / 1024
disk_fp_kb  = n * 4         / 1024

print("\n┌────────────────────────────┬──────────────┬─────────────┐")
print("│            model           │ disk (KB)    │ latency (ms)│")
print("├────────────────────────────┼──────────────┼─────────────┤")
print(f"│ FP32 MiniLLM (baseline)    │   {disk_fp_kb:>8.1f}   │   {t_fp:>7.2f}   │")
print(f"│ BitLinear MiniLLM (b1.58)  │   {disk_bit_kb:>8.1f}   │   {t_bit:>7.2f}   │")
print("├────────────────────────────┼──────────────┼─────────────┤")
print(f"│ compression / speedup      │   {disk_fp_kb / disk_bit_kb:>5.1f}×       │   {t_fp / t_bit:>5.2f}×     │")
print("└────────────────────────────┴──────────────┴─────────────┘")


## 7 · Sample generations

Tiny model, tiny corpus — but you can verify the generated stream has the same statistics as the training corpus (3-gram structure) rather than uniform random.


In [ ]:
@torch.no_grad()
def sample(model, n_new: int = 32, temperature: float = 0.8, seed: int = 0) -> list[int]:
    g = torch.Generator(device=device).manual_seed(seed)
    prompt = torch.randint(0, V, (1, 4), device=device, generator=g)
    return model.generate(prompt, max_new_tokens=n_new, temperature=temperature)[0].tolist()


print("BitLinear sample (first 24 tokens):", sample(model_bit, n_new=20)[:24])
print("FP32      sample (first 24 tokens):", sample(model_fp,  n_new=20)[:24])

# As a sanity check, the entropy of generated tokens should match the training corpus,
# not the uniform distribution.
@torch.no_grad()
def empirical_entropy(stream: list[int]) -> float:
    counts = torch.bincount(torch.tensor(stream), minlength=V).float()
    probs  = counts / counts.sum()
    nz = probs[probs > 0]
    return float((-nz * nz.log()).sum().item())


long_bit = sample(model_bit, n_new=500)
long_fp  = sample(model_fp,  n_new=500)
print(f"\nentropy of training corpus (sampled): {empirical_entropy(corpus[:500].tolist()):.3f}")
print(f"entropy of BitLinear continuation   : {empirical_entropy(long_bit):.3f}")
print(f"entropy of FP32      continuation   : {empirical_entropy(long_fp):.3f}")
print(f"entropy of uniform random           : {math.log(V):.3f}")


## 8 · Discussion

- **Ternary works at small scale.** With ~150k parameters and ~400 steps the BitLinear model tracks the FP32 baseline closely on a structured-language task. The published BitNet b1.58 results show this gap fully closes above ≈3B parameters.
- **The wins materialize on real hardware.** Wall-time speedups require either bit-packed kernels (BitBLAS, T-MAC) or sparse-aware silicon. On stock CUDA the BitLinear forward pays the dequantization cost without recouping it — that's why the latency column above can be ≥ 1×.
- **Where to go next.**
  - Plug a real corpus: switch `make_corpus` for a `datasets.load_dataset(...)` call against TinyStories or WikiText-103.
  - Enable YaRN by constructing the `MiniLLM` with `cfg.max_seq_len = 4×` and substituting `RotaryPositionalEmbedding` for `YaRNRotaryPositionalEmbedding` in `src/model/mini_llm.py`.
  - Train longer, then run `scripts/export_quantized.py` to produce the packed-ternary `.bwf` artifact for deployment.

### What you built across these six weeks

A reference codebase that touches **every layer** of the modern LLM stack — derived from first principles, implemented in raw PyTorch, unit-tested, and benchmark-instrumented. The same shapes scale (with appropriate hyper-parameters and engineering) to multi-billion-parameter models without changing the underlying math.
